# Generate Sample Data — Databricks Residential

Creates realistic property investment data for the demo:
- ~50 properties across 8 US metros
- ~18 months of rent records (~30K-80K rows)
- Story: Denver vacancy spike starting 6 months ago

Output: JSON files in UC Volume for Auto Loader ingestion.

## Setup: Create UC Infrastructure

In [ ]:
CATALOG = "startups_catalog"
RAW_SCHEMA = "raw"

# Catalog already exists — create schemas and volume
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}.data")
print(f"UC infrastructure ready: {CATALOG}")

## Generate Properties

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime, timedelta
import numpy as np

NUM_PROPERTIES = 50

# Markets with weighted distribution (Denver overweight for story)
MARKETS = [
    ("Denver", "CO", "MKT-DEN", 0.20),
    ("Austin", "TX", "MKT-AUS", 0.15),
    ("Nashville", "TN", "MKT-NSH", 0.12),
    ("Phoenix", "AZ", "MKT-PHX", 0.12),
    ("Tampa", "FL", "MKT-TPA", 0.10),
    ("Charlotte", "NC", "MKT-CLT", 0.10),
    ("Raleigh", "NC", "MKT-RAL", 0.08),
    ("Salt Lake City", "UT", "MKT-SLC", 0.07),
    ("Boise", "ID", "MKT-BOI", 0.06),
]

In [ ]:
import pandas as pd
from pyspark.sql.types import StringType

@F.pandas_udf(StringType())
def fake_property_name(ids: pd.Series) -> pd.Series:
    from faker import Faker
    fake = Faker()
    suffixes = ["Apartments", "Residences", "Lofts", "Flats", "Place", "Terrace", "Commons", "Heights", "Village", "Crossing"]
    import random
    return pd.Series([f"{fake.last_name()} {random.choice(suffixes)}" for _ in range(len(ids))])

@F.pandas_udf(StringType())
def fake_address(ids: pd.Series) -> pd.Series:
    from faker import Faker
    fake = Faker()
    return pd.Series([fake.street_address() for _ in range(len(ids))])

@F.pandas_udf(StringType())
def fake_zip(ids: pd.Series) -> pd.Series:
    from faker import Faker
    fake = Faker()
    return pd.Series([fake.zipcode() for _ in range(len(ids))])

In [ ]:
# Build cumulative weights for market assignment
market_thresholds = []
cumulative = 0.0
for city, state, market_id, weight in MARKETS:
    cumulative += weight
    market_thresholds.append((cumulative, city, state, market_id))

# Generate base properties
properties_df = spark.range(0, NUM_PROPERTIES, numPartitions=4).select(
    F.concat(F.lit("PROP-"), F.lpad(F.col("id").cast("string"), 5, "0")).alias("property_id"),
    fake_property_name(F.col("id")).alias("property_name"),
    fake_address(F.col("id")).alias("address"),
    F.rand(seed=42).alias("_market_rand"),
    F.rand(seed=99).alias("_type_rand"),
    F.rand(seed=77).alias("_class_rand"),
    F.col("id"),
)

# Assign market based on weighted distribution
market_expr = F.lit(MARKETS[-1][0])
state_expr = F.lit(MARKETS[-1][1])
market_id_expr = F.lit(MARKETS[-1][2])

for threshold, city, state, market_id in reversed(market_thresholds):
    market_expr = F.when(F.col("_market_rand") < threshold, city).otherwise(market_expr)
    state_expr = F.when(F.col("_market_rand") < threshold, state).otherwise(state_expr)
    market_id_expr = F.when(F.col("_market_rand") < threshold, market_id).otherwise(market_id_expr)

# Property type: 60% multifamily, 25% single_family, 15% mixed_use
type_expr = (
    F.when(F.col("_type_rand") < 0.60, "multifamily")
    .when(F.col("_type_rand") < 0.85, "single_family")
    .otherwise("mixed_use")
)

# Asset class: 20% A, 50% B, 30% C
class_expr = (
    F.when(F.col("_class_rand") < 0.20, "A")
    .when(F.col("_class_rand") < 0.70, "B")
    .otherwise("C")
)

# Resolve all rand-based expressions in this select so _market_rand/_type_rand/_class_rand are consumed
properties_df = properties_df.select(
    "property_id", "property_name", "address",
    market_expr.alias("city"),
    state_expr.alias("state"),
    fake_zip(F.col("id")).alias("zip_code"),
    type_expr.alias("property_type"),
    class_expr.alias("asset_class"),
    market_id_expr.alias("market_id"),
    F.col("id"),
)

In [ ]:
# Units: multifamily 30-250, single_family 1-4, mixed_use 10-60
units_expr = (
    F.when(F.col("property_type") == "multifamily",
           (F.abs(F.hash(F.col("id"))) % 221 + 30).cast("int"))
    .when(F.col("property_type") == "single_family",
           (F.abs(F.hash(F.col("id") + 1)) % 4 + 1).cast("int"))
    .otherwise((F.abs(F.hash(F.col("id") + 2)) % 51 + 10).cast("int"))
)

# Purchase price: log-normal by type
price_expr = (
    F.when(F.col("property_type") == "multifamily",
           F.round(F.exp(F.log(F.lit(8000000.0)) + F.randn(seed=42) * 0.5), 2))
    .when(F.col("property_type") == "single_family",
           F.round(F.exp(F.log(F.lit(450000.0)) + F.randn(seed=43) * 0.4), 2))
    .otherwise(F.round(F.exp(F.log(F.lit(3000000.0)) + F.randn(seed=44) * 0.5), 2))
)

properties_df = properties_df.select(
    "property_id", "property_name", "address", "city", "state", "zip_code",
    "property_type", "asset_class",
    units_expr.alias("units"),
    (F.col("id") * 0 + F.abs(F.hash(F.col("id") + 3)) % 50 + 1965).alias("year_built"),
    F.date_sub(F.current_date(), (F.abs(F.hash(F.col("id") + 4)) % 1800).cast("int")).alias("acquisition_date"),
    price_expr.alias("purchase_price"),
    "market_id",
    F.col("id"),
)

# Appraised value: purchase price * 1.0-1.3
properties_df = properties_df.withColumn(
    "current_appraised_value",
    F.round(F.col("purchase_price") * (1.0 + F.rand(seed=55) * 0.3), 2),
).withColumn(
    "square_footage",
    (F.col("units") * (F.abs(F.hash(F.col("id") + 5)) % 400 + 600)).cast("int"),
)

# Final columns
properties_df = properties_df.select(
    "property_id", "property_name", "address", "city", "state", "zip_code",
    "property_type", "asset_class", "units", "square_footage", "year_built",
    "acquisition_date", "purchase_price", "current_appraised_value", "market_id",
)

print(f"Properties count: {properties_df.count()}")
properties_df.show(5, truncate=False)

In [ ]:
# Write properties to Volume as JSON for Auto Loader
VOLUME_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/data"

properties_df.coalesce(1).write.mode("overwrite").json(f"{VOLUME_PATH}/properties/")
print(f"Properties written to {VOLUME_PATH}/properties/")

## Generate Rent Records

**Story:** Denver properties show a vacancy spike starting ~6 months ago.
- Denver occupancy drops from ~95% to ~80%
- Denver delinquency rate increases from ~5% to ~15%
- Other markets remain stable at ~93-96% occupancy

In [ ]:
# Write properties to Delta for FK lookup (no .cache() on serverless)
properties_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{RAW_SCHEMA}.properties_lookup")
prop_lookup = spark.table(f"{CATALOG}.{RAW_SCHEMA}.properties_lookup").select(
    "property_id", "city", "units", "asset_class"
)

# Generate 18 months of dates
NUM_MONTHS = 18
today = datetime.now()
start_month = today - timedelta(days=NUM_MONTHS * 30)
month_dates = [
    (start_month + timedelta(days=30 * i)).replace(day=1).strftime("%Y-%m-%d")
    for i in range(NUM_MONTHS)
]

months_df = spark.createDataFrame(
    [(i, d) for i, d in enumerate(month_dates)],
    schema="month_idx INT, rent_date STRING"
)

# Cross join: one row per property x month
prop_months = prop_lookup.crossJoin(months_df)

# Explode into individual unit rows
prop_months_units = prop_months.withColumn(
    "unit_number",
    F.explode(F.sequence(F.lit(1), F.col("units")))
).withColumn(
    "unit_number",
    F.concat(F.lit("U"), F.lpad(F.col("unit_number").cast("string"), 4, "0"))
)

# Generate rent_id
prop_months_units = prop_months_units.withColumn(
    "rent_id",
    F.concat(
        F.col("property_id"), F.lit("-"),
        F.col("unit_number"), F.lit("-"),
        F.regexp_replace(F.col("rent_date"), "-", "")
    )
)

print(f"Total unit-month rows: {prop_months_units.count()}")

In [ ]:
# Occupancy: Denver spike in last 6 months
is_denver = F.col("city") == "Denver"
is_recent = F.col("month_idx") >= 12  # Last 6 months
vacancy_rand = F.rand(seed=123)

base_vacancy = (
    F.when(F.col("asset_class") == "A", 0.05)
    .when(F.col("asset_class") == "B", 0.07)
    .otherwise(0.10)
)
denver_spike_vacancy = (
    F.when(F.col("asset_class") == "A", 0.20)
    .when(F.col("asset_class") == "B", 0.25)
    .otherwise(0.30)
)
vacancy_rate = F.when(is_denver & is_recent, denver_spike_vacancy).otherwise(base_vacancy)

prop_months_units = prop_months_units.withColumn("is_occupied", vacancy_rand > vacancy_rate)

# Tenant ID: only for occupied units
prop_months_units = prop_months_units.withColumn(
    "tenant_id",
    F.when(F.col("is_occupied"),
           F.concat(F.lit("TNT-"), F.abs(F.hash(F.concat(F.col("property_id"), F.col("unit_number")))).cast("string")))
    .otherwise(None)
)

# Monthly rent by asset class: A ~$2200, B ~$1500, C ~$1000
base_rent = (
    F.when(F.col("asset_class") == "A",
           F.round(F.exp(F.log(F.lit(2200.0)) + F.randn(seed=200) * 0.15), 2))
    .when(F.col("asset_class") == "B",
           F.round(F.exp(F.log(F.lit(1500.0)) + F.randn(seed=201) * 0.15), 2))
    .otherwise(
           F.round(F.exp(F.log(F.lit(1000.0)) + F.randn(seed=202) * 0.15), 2))
)
prop_months_units = prop_months_units.withColumn("monthly_rent", base_rent)

# Rent collected: delinquency ~5% normal, ~15% Denver recent
delinquency_rand = F.rand(seed=456)
delinquency_rate = F.when(is_denver & is_recent, 0.15).otherwise(0.05)

prop_months_units = prop_months_units.withColumn(
    "rent_collected",
    F.when(~F.col("is_occupied"), 0.0)
    .when(delinquency_rand < delinquency_rate,
          F.round(F.col("monthly_rent") * F.rand(seed=789) * 0.5, 2))
    .otherwise(F.col("monthly_rent"))
)

# Lease dates (12-month leases)
prop_months_units = prop_months_units.withColumn(
    "lease_start_date",
    F.when(F.col("is_occupied"),
           F.date_sub(F.col("rent_date").cast("date"),
                      (F.abs(F.hash(F.concat(F.col("rent_id"), F.lit("ls")))) % 365).cast("int")))
    .otherwise(None)
).withColumn(
    "lease_end_date",
    F.when(F.col("is_occupied"), F.date_add(F.col("lease_start_date"), 365))
    .otherwise(None)
)

In [ ]:
# Select final rent columns and write to Volume
rents_df = prop_months_units.select(
    "rent_id", "property_id", "unit_number", "tenant_id",
    "lease_start_date", "lease_end_date",
    "monthly_rent", "rent_collected", "is_occupied",
    F.col("rent_date").cast("date").alias("rent_date"),
)

print(f"Rent records count: {rents_df.count()}")
rents_df.repartition(6).write.mode("overwrite").json(f"{VOLUME_PATH}/rents/")
print(f"Rents written to {VOLUME_PATH}/rents/")

## Validate Output

In [ ]:
print("=== Properties ===")
props_check = spark.read.json(f"{VOLUME_PATH}/properties/")
print(f"Count: {props_check.count()}")
props_check.groupBy("city").count().orderBy("count", ascending=False).show()
props_check.groupBy("property_type").count().show()
props_check.groupBy("asset_class").count().show()

print("\n=== Rents ===")
rents_check = spark.read.json(f"{VOLUME_PATH}/rents/")
print(f"Count: {rents_check.count()}")

In [ ]:
# Denver vacancy story validation
print("=== Denver Vacancy Story ===")
denver_props = props_check.filter(F.col("city") == "Denver").select("property_id")
denver_rents = rents_check.join(denver_props, on="property_id")

denver_rents.groupBy("rent_date").agg(
    F.round(F.avg(F.col("is_occupied").cast("int")) * 100, 1).alias("occupancy_pct"),
    F.round(F.avg(F.col("rent_collected")), 2).alias("avg_collected"),
).orderBy("rent_date").show(20)

In [ ]:
# Cleanup temp lookup table
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{RAW_SCHEMA}.properties_lookup")
print("Done! Sample data generated and written to UC Volume.")